In [2]:
from preprocessing import create_features, get_data, get_encoded_data, split_data

features, target = get_data()
features = create_features(features)
encoded_features, target = get_encoded_data(features, target)
X_train, X_valid, X_test, y_train, y_valid, y_test = split_data(encoded_features, target)

In [3]:
from lightgbm import LGBMClassifier

lgbm_model = LGBMClassifier(
    random_state=42,
    force_col_wise=True,
    max_depth=10,
    num_leaves=62,
    n_estimators=140,
    learning_rate=0.01,
    # is_unbalance=True,
    # class_weight='balanced',
    scale_pos_weight=5,
)
lgbm_model.fit(X_train, y_train)
lgbm_valid_pred = lgbm_model.predict(X_valid)
lgbm_valid_pred_proba = lgbm_model.predict_proba(X_valid)[:, 1]

[LightGBM] [Info] Number of positive: 1154, number of negative: 4471
[LightGBM] [Info] Total Bins 1377
[LightGBM] [Info] Number of data points in the train set: 5625, number of used features: 19
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.205156 -> initscore=-1.354378
[LightGBM] [Info] Start training from score -1.354378


In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=850,
    max_depth=3,
    learning_rate=0.01,
    scale_pos_weight=4.5,
    random_state=42,
    objective='binary:logistic',
    tree_method='approx',
    )
xgb_model.fit(X_train, y_train)
xgb_valid_pred = xgb_model.predict(X_valid)
xgb_valid_pred_proba = xgb_model.predict_proba(X_valid)[:, 1]

In [20]:
ensemble_valid_pred_proba = (lgbm_valid_pred_proba + xgb_valid_pred_proba) / 2
ensemble_valid_pred = (ensemble_valid_pred_proba >= 0.48).astype(int)

In [21]:
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

accuracy = accuracy_score(y_valid, ensemble_valid_pred)
f1 = f1_score(y_valid, ensemble_valid_pred)
recall = recall_score(y_valid, ensemble_valid_pred)
precision = precision_score(y_valid, ensemble_valid_pred)

print(f"Validation 성능 - Accuracy: {accuracy:.4f}, F1 Score: {f1:.4f}, Recall: {recall:.4f}, Precision: {precision:.4f}")

Validation 성능 - Accuracy: 0.7979, F1 Score: 0.6064, Recall: 0.7565, Precision: 0.5061
